# 01 — PEERS Dataset Exploration

**Purpose:** Explore the PEERS (Penn Electrophysiology of Encoding and Retrieval Study) dataset.
Visualize raw EEG signals, check data quality, understand event structure, and examine
the distribution of recalled vs. forgotten items.

**Dataset:** OpenNeuro ds004106 — Free recall EEG in BIDS format.

In [ ]:
import sys
sys.path.insert(0, "../..")

import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from shared.preprocessing.bids import get_bids_subjects, read_bids_raw
from shared.preprocessing.pipeline import PreprocessingConfig, preprocess_raw

# Configure
PEERS_DIR = Path("../../data/raw/peers")
mne.set_log_level("WARNING")
plt.style.use("dark_background")

## 1. Dataset Overview

List available subjects and check BIDS structure.

In [ ]:
# List subjects in the BIDS dataset
subjects = get_bids_subjects(PEERS_DIR)
print(f"Number of subjects: {len(subjects)}")
print(f"Subject IDs: {subjects[:10]}{'...' if len(subjects) > 10 else ''}")

# Check for participants.tsv (demographics)
participants_file = PEERS_DIR / "participants.tsv"
if participants_file.exists():
    import pandas as pd
    participants = pd.read_csv(participants_file, sep="\t")
    print(f"\nParticipants table:")
    display(participants.head(10))
    print(f"\nColumns: {list(participants.columns)}")
else:
    print("\nNo participants.tsv found")

## 2. Load and Inspect One Subject

Examine raw EEG: channels, sampling rate, duration, events.

In [ ]:
# Load first subject
subject = subjects[0]
raw = read_bids_raw(PEERS_DIR, subject=subject, task="encoding")

print(f"Subject: {subject}")
print(f"Channels: {len(raw.ch_names)}")
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Duration: {raw.times[-1]:.1f} seconds ({raw.times[-1]/60:.1f} min)")
print(f"Channel types: {set(raw.get_channel_types())}")
print(f"\nFirst 10 channels: {raw.ch_names[:10]}")
print(f"Bad channels: {raw.info['bads']}")

## 3. Raw Signal Visualization

Plot a short segment of raw EEG to see signal quality.

In [ ]:
# Plot 5 seconds of raw data (first 8 channels)
fig, axes = plt.subplots(8, 1, figsize=(14, 10), sharex=True)
start_sample = int(10 * raw.info["sfreq"])  # start at 10s
n_samples = int(5 * raw.info["sfreq"])      # show 5s

data = raw.get_data(start=start_sample, stop=start_sample + n_samples)
times = np.arange(n_samples) / raw.info["sfreq"]

for i, ax in enumerate(axes):
    if i < len(raw.ch_names):
        ax.plot(times, data[i] * 1e6, linewidth=0.5, color="#4FC3F7")
        ax.set_ylabel(raw.ch_names[i], fontsize=8, rotation=0, ha="right")
        ax.set_ylim(-100, 100)
        ax.tick_params(labelsize=7)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Raw EEG — Subject {subject} (5s segment)", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Event Structure

Examine the encoding events — what stimulus types are present,
how many trials per condition.

In [ ]:
# Extract events
events, event_id = mne.events_from_annotations(raw)
print(f"Total events: {len(events)}")
print(f"\nEvent types:")
for name, code in sorted(event_id.items(), key=lambda x: x[1]):
    count = (events[:, 2] == code).sum()
    print(f"  {name} (code={code}): {count} events")

# Plot event timeline
fig, ax = plt.subplots(figsize=(14, 3))
for name, code in event_id.items():
    mask = events[:, 2] == code
    event_times = events[mask, 0] / raw.info["sfreq"]
    ax.scatter(event_times, [code] * mask.sum(), s=3, label=name, alpha=0.7)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Event code")
ax.set_title("Event timeline")
ax.legend(bbox_to_anchor=(1.02, 1), fontsize=7)
plt.tight_layout()
plt.show()

## 5. Power Spectral Density

Check the frequency content — do we see expected alpha peak (~10 Hz)?

In [ ]:
# Compute and plot PSD
fig, ax = plt.subplots(figsize=(10, 5))
spectrum = raw.compute_psd(fmin=0.5, fmax=45, n_fft=1024)
spectrum.plot(axes=ax, show=False, color="#4FC3F7", spatial_colors=True)

# Mark frequency bands
band_colors = {"δ 0.5-4": "#666", "θ 4-8": "#888", "α 8-13": "#AAA", "β 13-30": "#888", "γ 30-45": "#666"}
bands = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
for (fmin, fmax), (label, color) in zip(bands, band_colors.items()):
    ax.axvspan(fmin, fmax, alpha=0.1, color=color)
    ax.text((fmin + fmax) / 2, ax.get_ylim()[1] * 0.95, label,
            ha="center", fontsize=8, color="#CCC")

ax.set_title(f"Power Spectral Density — Subject {subject}")
plt.tight_layout()
plt.show()

## 6. Recall Distribution

How many items were recalled vs. forgotten across subjects?
This is the label balance for our classifier.

In [ ]:
# Load data with our loader to get labels
from classifiers.encoding.data import load_subject_data, PEERSConfig

config = PEERSConfig(bids_root=PEERS_DIR)
subject_data = load_subject_data(PEERS_DIR, subject=subjects[0], config=config)

if subject_data is not None:
    labels = subject_data.labels
    n_recalled = labels.sum()
    n_forgotten = (1 - labels).sum()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    # Bar chart
    bars = ax1.bar(["Recalled", "Forgotten"], [n_recalled, n_forgotten],
                   color=["#4FC3F7", "#E57373"], edgecolor="#333", linewidth=2)
    ax1.set_ylabel("Number of trials")
    ax1.set_title(f"Subject {subjects[0]} — Recall Distribution")
    for bar, val in zip(bars, [n_recalled, n_forgotten]):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(int(val)), ha="center", fontsize=11)

    # Pie chart
    ax2.pie([n_recalled, n_forgotten], labels=["Recalled", "Forgotten"],
            colors=["#4FC3F7", "#E57373"], autopct="%1.1f%%",
            textprops={"color": "#EEE"})
    ax2.set_title("Label Balance")

    plt.tight_layout()
    plt.show()

    print(f"\nRecalled: {int(n_recalled)} ({n_recalled/len(labels)*100:.1f}%)")
    print(f"Forgotten: {int(n_forgotten)} ({n_forgotten/len(labels)*100:.1f}%)")
else:
    print("Could not load subject data. Ensure PEERS is downloaded.")